# Evaluating emotion2vec performance on Savee datatet

In [ ]:
!pip install -U funasr modelscope

In [28]:
# Importing libraries
import pandas as pd
import os
import torch
from funasr import AutoModel
from google.colab import files, drive
import time
import wave
import numpy as np
from scipy.io.wavfile import write
import tempfile

In [ ]:
# Mount google drive
drive.mount('/content/drive')

In [30]:
# Load dataset from google drive storage
df = pd.read_parquet('/content/drive/MyDrive/savee.parquet')
print(df.head())
print(df['emotion'].unique())

         file                                              audio gender  \
0  DC_a01.wav  {'bytes': b'RIFF\x1e\xc8\x01\x00WAVEfmt \x10\x...   male   
1  DC_a02.wav  {'bytes': b'RIFF\xea\xad\x01\x00WAVEfmt \x10\x...   male   
2  DC_a03.wav  {'bytes': b'RIFF\x94\x03\x01\x00WAVEfmt \x10\x...   male   
3  DC_a04.wav  {'bytes': b'RIFF\xd0T\x01\x00WAVEfmt \x10\x00\...   male   
4  DC_a05.wav  {'bytes': b'RIFF\xe2v\x01\x00WAVEfmt \x10\x00\...   male   

                                       transcription emotion  speaking_rate  \
0  She had her dark suit in greasy wash water all...   anger          11.79   
1    "'Don't ask me to carry an oily rag like that.'   anger          12.22   
2                              Will you tell me why?   anger           7.71   
3      Who authorised the unlimited expense account?   anger          13.21   
4           Destroy every file related to my audits.   anger          11.67   

   pitch_mean  pitch_std       rms  relative_db  
0  169.301239  27.078539

In [31]:
def postProcessing(scores : np.ndarray, labels : np.ndarray):
    """ Extract emotion through scores returned by emotion2vec, merging neutral and other as a single category

    Args:
        scores (np.ndarray): scores generated by audio2emotion
        labels (np.ndarray): emotion labels from which to select
    """

    # Concatenate neutral, other and unk scores into a single column
    neutral = scores[4] + scores[5] + scores[8]
    scores = np.delete(scores, [4, 5, 8])
    scores = np.append(scores, [neutral])

    # Find the label corresponding to the maximum score
    max_index = np.argmax(scores)
    max_label = labels[max_index]       # Label produced through emotion2vec

    # Map the label to the corresponding one in the dataset
    mapping = {
        "angry" : "anger",
        "disgusted" : "disgust",
        "fearful" : "fear",
        "happy" : "happiness",
        "neutral" : "neutral",
        "surprised" : "surprise",
        "sad" : "sadness"
    }
    return mapping[max_label]

In [32]:
# Check GPU
print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")


# Load the models
model_ids = ["iic/emotion2vec_plus_seed", "iic/emotion2vec_plus_base", "iic/emotion2vec_plus_large"]

modelSeed = AutoModel(
    model = model_ids[0],
    hub = "hf",  # "ms" or "modelscope" for China mainland users; "hf" or "huggingface" for other overseas users
    disable_update = True,
    device = "cuda"
)

modelBase = AutoModel(
    model = model_ids[1],
    hub = "hf",
    disable_update = True,
    device = "cuda"
)

modelLarge = AutoModel(
    model = model_ids[2],
    hub = "hf",
    disable_update = True,
    device = "cuda"
)

# Extract the emotion labels
emotions = df['emotion']

# Labels returned by emotion2vec (without unk or other)
labels = np.array(['angry', 'disgusted', 'fearful', 'happy', 'sad', 'surprised', 'neutral'])

# Loop through the dataset entries
emotionsList1 = []
emotionsList2 = []
emotionsList3 = []
timeList1 = []
timeList2 = []
timeList3 = []
timePostList1 = []
timePostList2 = []
timePostList3 = []
for i, row in df.iterrows():
    print("\n ================================================== \n ")
    print(f"\n Starting processing of row {i} \n")
    data = row['audio']['bytes']

    with tempfile.NamedTemporaryFile(suffix = ".wav", mode = "wb") as file:
        # Save the audio binary raw data
        file.write(data)
        path = file.name

        # Perform emotion recognition using seed model
        start = time.perf_counter()
        print(f"{torch.cuda.memory_allocated()} \n")        # Check GPU is used
        result = modelSeed.generate(path, language = "en", extract_embedding = False)
        print(f"{torch.cuda.memory_allocated()} \n")        # Check GPU is used
        end = time.perf_counter()
        print(f"\n Time for emotion detection with model {model_ids[0]} is {end - start} \n")
        timeList1.append(end - start)

        # Post processing of seed model result
        start = time.perf_counter()
        emotion1 = postProcessing(np.array(result[0]['scores'], dtype = float), labels)
        emotionsList1.append(emotion1)
        end = time.perf_counter()
        print(f"\n Time for post processing with model {model_ids[0]} is {end - start} \n")
        timePostList1.append(end - start)

        # Perform emotion recognition using base model
        start = time.perf_counter()
        result = modelSeed.generate(path, language = "en", extract_embedding = False)
        end = time.perf_counter()
        print(f"\n Time for emotion detection with model {model_ids[1]} is {end - start} with label {emotion1} \n")
        timeList2.append(end - start)

        # Post processing of base model result
        start = time.perf_counter()
        emotion2 = postProcessing(np.array(result[0]['scores'], dtype = float), labels)
        emotionsList2.append(emotion2)
        end = time.perf_counter()
        print(f"\n Time for post processing with model {model_ids[1]} is {end - start} with label {emotion2} \n")
        timePostList2.append(end - start)

        # Perform emotion recognition using large model
        start = time.perf_counter()
        result = modelSeed.generate(path, language = "en", extract_embedding = False)
        end = time.perf_counter()
        print(f"\n Time for emotion detection with model {model_ids[2]} is {end - start} \n")
        timeList3.append(end - start)

        # Post processing of base model result
        start = time.perf_counter()
        emotion3 = postProcessing(np.array(result[0]['scores'], dtype = float), labels)
        emotionsList3.append(emotion3)
        end = time.perf_counter()
        print(f"\n Time for post processing with model {model_ids[2]} is {end - start} with label {emotion3} \n")
        timePostList3.append(end - start)

        # Clean the temporary file
        file.flush()


True
1
Tesla T4
funasr version: 1.3.14.


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Warning, miss key in ckpt: modality_encoders.AUDIO.decoder.blocks.0.0.weight, /root/.cache/huggingface/hub/models--emotion2vec--emotion2vec_plus_seed/snapshots/7be9a30d52bdc793cc24e6f7d25358d6648927c1/model.pt
Warning, miss key in ckpt: modality_encoders.AUDIO.decoder.blocks.0.0.bias, /root/.cache/huggingface/hub/models--emotion2vec--emotion2vec_plus_seed/snapshots/7be9a30d52bdc793cc24e6f7d25358d6648927c1/model.pt
Warning, miss key in ckpt: modality_encoders.AUDIO.decoder.blocks.1.0.weight, /root/.cache/huggingface/hub/models--emotion2vec--emotion2vec_plus_seed/snapshots/7be9a30d52bdc793cc24e6f7d25358d6648927c1/model.pt
Warning, miss key in ckpt: modality_encoders.AUDIO.decoder.blocks.1.0.bias, /root/.cache/huggingface/hub/models--emotion2vec--emotion2vec_plus_seed/snapshots/7be9a30d52bdc793cc24e6f7d25358d6648927c1/model.pt
Warning, miss key in ckpt: modality_encoders.AUDIO.decoder.blocks.2.0.weight, /root/.cache/huggingface/hub/models--emotion2vec--emotion2vec_plus_seed/snapshots/7be9

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Warning, miss key in ckpt: modality_encoders.AUDIO.decoder.blocks.0.0.weight, /root/.cache/huggingface/hub/models--emotion2vec--emotion2vec_plus_base/snapshots/b318240bfe67db81a8c572ecb37ce9c3759b81c9/model.pt
Warning, miss key in ckpt: modality_encoders.AUDIO.decoder.blocks.0.0.bias, /root/.cache/huggingface/hub/models--emotion2vec--emotion2vec_plus_base/snapshots/b318240bfe67db81a8c572ecb37ce9c3759b81c9/model.pt
Warning, miss key in ckpt: modality_encoders.AUDIO.decoder.blocks.1.0.weight, /root/.cache/huggingface/hub/models--emotion2vec--emotion2vec_plus_base/snapshots/b318240bfe67db81a8c572ecb37ce9c3759b81c9/model.pt
Warning, miss key in ckpt: modality_encoders.AUDIO.decoder.blocks.1.0.bias, /root/.cache/huggingface/hub/models--emotion2vec--emotion2vec_plus_base/snapshots/b318240bfe67db81a8c572ecb37ce9c3759b81c9/model.pt
Warning, miss key in ckpt: modality_encoders.AUDIO.decoder.blocks.2.0.weight, /root/.cache/huggingface/hub/models--emotion2vec--emotion2vec_plus_base/snapshots/b318

...

1437683712 


 Time for emotion detection with model iic/emotion2vec_plus_seed is 0.07481120499960525 


 Time for post processing with model iic/emotion2vec_plus_seed is 0.00011936499959119828 



rtf_avg: 0.008: 100%|██████████| 1/1 [00:00<00:00, 19.23it/s]



 Time for emotion detection with model iic/emotion2vec_plus_base is 0.05469841600006475 with label surprise 


 Time for post processing with model iic/emotion2vec_plus_base is 9.363799927086802e-05 with label surprise 



rtf_avg: 0.008: 100%|██████████| 1/1 [00:00<00:00, 18.73it/s]



 Time for emotion detection with model iic/emotion2vec_plus_large is 0.05652230600026087 


 Time for post processing with model iic/emotion2vec_plus_large is 0.0001149159998021787 with label surprise 



In [36]:
# Generate Pandas Series with the emotions labels generated
emotionsSeries1 = pd.Series(emotionsList1)
emotionsSeries2 = pd.Series(emotionsList2)
emotionsSeries3 = pd.Series(emotionsList3)

# Compare the generated labels with the true labels
comparison1 = emotions == emotionsSeries1
comparison2 = emotions == emotionsSeries2
comparison3 = emotions == emotionsSeries3

# Generate Pandas Series with the times generated
timeSeries1 = pd.Series(timeList1)
timeSeries2 = pd.Series(timeList2)
timeSeries3 = pd.Series(timeList3)

timePostSeries1 = pd.Series(timePostList1)
timePostSeries2 = pd.Series(timePostList2)
timePostSeries3 = pd.Series(timePostList3)

# Store the time statistics for each model along with summary statistics
times = pd.DataFrame({
    'Seed' : timeSeries1.describe(),
    'Base' : timeSeries2.describe(),
    'Large' : timeSeries3.describe(),
    'Seed post' : timePostSeries1.describe(),
    'Base post' : timePostSeries2.describe(),
    'Large post' : timePostSeries3.describe()

})

times.to_csv(path_or_buf = f"/content/drive/My Drive/times.csv")

stats1 = timeSeries1.describe()
stats1.to_csv(path_or_buf = f"/content/drive/My Drive/timeStatsSeed.csv")
stats2 = timeSeries2.describe()
stats2.to_csv(path_or_buf = f"/content/drive/My Drive/timeStatsBase.csv")
stats3 = timeSeries3.describe()
stats3.to_csv(path_or_buf = f"/content/drive/My Drive/timeStatsLarge.csv")

statsPost1 = timePostSeries1.describe()
statsPost1.to_csv(path_or_buf = f"/content/drive/My Drive/timeStatsPostSeed.csv")
statsPost2 = timePostSeries2.describe()
statsPost2.to_csv(path_or_buf = f"/content/drive/My Drive/timeStatsPostBase.csv")
statsPost3 = timePostSeries3.describe()
statsPost3.to_csv(path_or_buf = f"/content/drive/My Drive/timeStatsPostLarge.csv")

# Store frames with the emotions generated in comparison to the true ones
emotions1 = pd.DataFrame({
    'true' : emotions,
    'predicted' : emotionsSeries1,
    'correct' : comparison1
})
emotions2 = pd.DataFrame({
    'true' : emotions,
    'predicted' : emotionsSeries2,
    'correct' : comparison2
})
emotions3 = pd.DataFrame({
    'true' : emotions,
    'predicted' : emotionsSeries3,
    'correct' : comparison3
})
emotions1.to_csv(path_or_buf = f"/content/drive/My Drive/emotions1.csv")
emotions2.to_csv(path_or_buf = f"/content/drive/My Drive/emotions2.csv")
emotions3.to_csv(path_or_buf = f"/content/drive/My Drive/emotions3.csv")

# Store the number of correct answers
correct1 = comparison1.sum()
correct2 = comparison2.sum()
correct3 = comparison3.sum()
correct = pd.DataFrame({
    'model': ['seed', 'base', 'large'],
    'accuracy': [
        comparison1.mean(),
        comparison2.mean(),
        comparison3.mean()
    ]
})
correct.to_csv(path_or_buf = f"/content/drive/My Drive/correct.csv")

In [37]:
# Download the files uploaded in Google Drive in local storage
files.download("/content/drive/My Drive/timeStatsSeed.csv")
files.download("/content/drive/My Drive/timeStatsBase.csv")
files.download("/content/drive/My Drive/timeStatsLarge.csv")
files.download("/content/drive/My Drive/timeStatsPostSeed.csv")
files.download("/content/drive/My Drive/timeStatsPostBase.csv")
files.download("/content/drive/My Drive/timeStatsPostLarge.csv")
files.download("/content/drive/My Drive/emotions1.csv")
files.download("/content/drive/My Drive/emotions2.csv")
files.download("/content/drive/My Drive/emotions3.csv")
files.download("/content/drive/My Drive/times.csv")
files.download("/content/drive/My Drive/correct.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>